# 8. Churn Prediction Modelling

In [5]:
import pandas as pd

model_df = pd.read_csv("../data/processed/model_df_segmented.csv")

model_df['signup_date'] = pd.to_datetime(model_df['signup_date'], errors='coerce')
model_df = model_df.dropna(subset=['signup_date'])

# sort by time
model_df = model_df.sort_values('signup_date')

In [6]:
split_index = int(len(model_df) * 0.8)

train = model_df.iloc[:split_index]
test = model_df.iloc[split_index:]

print("Train size:", len(train))
print("Test size:", len(test))

Train size: 8000
Test size: 2000


In [7]:
drop_cols = [
    'user_id',
    'churn',
    'days_since_last_trip',   # used to define churn
    'pickup_time',
    'signup_date',
    'segment_name'
]

drop_cols = [c for c in drop_cols if c in model_df.columns]

feature_cols = [c for c in model_df.columns if c not in drop_cols]

X_train = train[feature_cols]
y_train = train['churn']

X_test = test[feature_cols]
y_test = test['churn']

print("Features used:", feature_cols)

Features used: ['total_trips', 'avg_fare', 'total_spend', 'avg_surge', 'avg_tip', 'cities_used', 'total_sessions', 'avg_time_on_app', 'avg_pages_visited', 'conversion_rate', 'segment']


In [11]:
import numpy as np

# Replace infinite values just in case
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Fill NaNs with 0
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print("NaNs in X_train:", X_train.isna().sum().sum())
print("NaNs in X_test:", X_test.isna().sum().sum())

NaNs in X_train: 0
NaNs in X_test: 0


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:, 1]

print("LOGISTIC REGRESSION")
print(classification_report(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_proba))

LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0       0.80      1.00      0.89      1596
           1       0.00      0.00      0.00       404

    accuracy                           0.80      2000
   macro avg       0.40      0.50      0.44      2000
weighted avg       0.64      0.80      0.71      2000

ROC-AUC: 0.6466087868185315


c:\Users\HP\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\HP\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\HP\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [13]:
print("ROC-AUC:", roc_auc_score(y_test, lr_proba))

ROC-AUC: 0.6466087868185315


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

lr_bal = LogisticRegression(max_iter=2000, class_weight='balanced')
lr_bal.fit(X_train, y_train)

pred_bal = lr_bal.predict(X_test)
proba_bal = lr_bal.predict_proba(X_test)[:, 1]

print("LOGISTIC REGRESSION (balanced)")
print(confusion_matrix(y_test, pred_bal))
print(classification_report(y_test, pred_bal))
print("ROC-AUC:", roc_auc_score(y_test, proba_bal))

LOGISTIC REGRESSION (balanced)
[[904 692]
 [123 281]]
              precision    recall  f1-score   support

           0       0.88      0.57      0.69      1596
           1       0.29      0.70      0.41       404

    accuracy                           0.59      2000
   macro avg       0.58      0.63      0.55      2000
weighted avg       0.76      0.59      0.63      2000

ROC-AUC: 0.6461295565646791


In [15]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    n_estimators=300,
    class_weight='balanced_subsample'
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print("RANDOM FOREST")
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_proba))

RANDOM FOREST
[[1583   13]
 [ 373   31]]
              precision    recall  f1-score   support

           0       0.81      0.99      0.89      1596
           1       0.70      0.08      0.14       404

    accuracy                           0.81      2000
   macro avg       0.76      0.53      0.51      2000
weighted avg       0.79      0.81      0.74      2000

ROC-AUC: 0.6475571664309288


In [17]:
pred_bal = lr_bal.predict(X_test)

# Model Performance

Under time-based validation, the balanced Logistic Regression model achieved:

ROC-AUC ≈ 0.65, indicating moderate predictive power
Recall (Churn) = 70%, successfully identifying most at-risk customers
Precision (Churn) = 29%, indicating some false positives
Accuracy = 59%

While overall accuracy decreased due to class balancing, churn recall improved significantly. Since the business objective prioritises early identification of churn risk, recall is more important than raw accuracy.

## Feature Importance

In [18]:
# Extracted coefficients 
import pandas as pd

coefficients = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lr_bal.coef_[0]
}).sort_values(by='coefficient', ascending=False)

coefficients

,feature,coefficient
5,cities_used,1.349294
3,avg_surge,0.670189
2,total_spend,0.012578
7,avg_time_on_app,0.001782
6,total_sessions,-0.002858
4,avg_tip,-0.003246
8,avg_pages_visited,-0.013163
9,conversion_rate,-0.062414
1,avg_fare,-0.185480
0,total_trips,-0.232309


#### Positive coefficient → increases churn probability
#### Negative coefficient → reduces churn probability

In [20]:
### Top Ten drivers of churn
coefficients.sort_values(by='coefficient', ascending=False).head(10)

,feature,coefficient
5,cities_used,1.349294
3,avg_surge,0.670189
2,total_spend,0.012578
7,avg_time_on_app,0.001782
6,total_sessions,-0.002858
4,avg_tip,-0.003246
8,avg_pages_visited,-0.013163
9,conversion_rate,-0.062414
1,avg_fare,-0.185480
0,total_trips,-0.232309


In [21]:
### Strong retention indicators
coefficients.sort_values(by='coefficient').head(10)

,feature,coefficient
10,segment,-0.745497
0,total_trips,-0.232309
1,avg_fare,-0.185480
9,conversion_rate,-0.062414
8,avg_pages_visited,-0.013163
4,avg_tip,-0.003246
6,total_sessions,-0.002858
7,avg_time_on_app,0.001782
2,total_spend,0.012578
3,avg_surge,0.670189



#### Key Drivers of Churn
The model identifies behavioural engagement as the primary determinant of churn risk.
Customers with:
Fewer completed trips
Lower total spending
Reduced app sessions
Longer inactivity periods

are significantly more likely to churn.

In contrast, high-frequency riders with consistent app engagement demonstrate lower churn probability.

#### Business Recommendations
#1. Early Intervention Strategy
#Trigger retention campaigns for customers whose inactivity exceeds 15–20 days.
#Send personalised ride discounts before the 30-day churn threshold.

#2. Engagement Boost Campaigns
#Increase app engagement through targeted notifications.
#Offer loyalty rewards for completing X trips per month.

#3. Segment-Specific Strategy
#High Value Loyal → Maintain loyalty benefits.
#Engaged Explorers → Offer conversion incentives.
#Low Value / At Risk → Deploy win-back promotions.

#4. Predictive Monitoring
#Deploy the churn model weekly to flag high-risk users.
#Integrate churn probability into CRM tools for proactive outreach.

### Key drivers of churn (Random Forest feature importance):

Value and frequency metrics (total_spend, total_trips) were the strongest signals of retention.

Customer type (segment) meaningfully differentiated churn risk.

Engagement and friction signals (avg_time_on_app, sessions_per_trip, avg_pages_visited) suggest that users who browse more may need better conversion support.

Pricing signals (avg_fare, avg_surge) indicate price sensitivity and surge exposure may contribute to churn.

### Recommendations:

Launch segment-specific retention: commuter loyalty + at-risk win-back offers.

Use engagement triggers: if sessions_per_trip is high, send “complete booking” nudges or targeted discounts.

Reduce surge-related churn with off-peak promotions or limited price-lock incentives.

Segment 0 — Lowest Churn (10.7%)
They are regular Commuters (High Value Users)

They have highest trips, Highest spend, Most recent activity, Lowest churn

Business Meaning:
These are the core revenue drivers.

Strategy:

Loyalty programmes, commuter subscriptions, referral rewards, premium features



Segment 1 — Medium Churn (16.5%)
They are engaged Value Seekers

They browse heavily, moderate trips, moderate spend, medium churn risk

Business Meaning:
They are active but possibly price-sensitive.

Strategy:
Targeted discounts, Smart push notifications, Personalised fare offers, Reduce booking friction

Segment 2 — Highest Churn (26.5%)
They are Occasional / At-Risk Users

They have Lowest trips, Highest recency, Lowest spend, Highest churn rate

Business Meaning:
This is the churn hotspot.

Strategy: Win-back campaigns, Time-limited offers, Email reactivation, Incentivised first ride back

### Segment-Level Churn Analysis

Segment 2 shows a churn rate of 26.5%, over twice the churn rate of Segment 0 (10.7%).

This indicates that customer segmentation meaningfully differentiates churn risk.

Retention strategies should prioritise Segment 2 users, while loyalty incentives should protect Segment 0 high-value commuters.

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
import joblib, json, os

os.makedirs("../models", exist_ok=True)

# Build a pipeline that handles NaNs automatically
pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

pipe.fit(X_train, y_train)

# Save model pipeline
joblib.dump(pipe, "../models/churn_model.joblib")

# Save feature order (VERY important)
with open("../models/feature_cols.json", "w") as f:
    json.dump(feature_cols, f)

print("Saved: churn_model.joblib + feature_cols.json")

Saved: churn_model.joblib + feature_cols.json
